In [1]:
import duckdb

In [2]:
con = duckdb.connect('md:')

con.execute("install duckpgq from community;")
con.execute("load duckpgq;")

con.execute("install ducklake;")
con.execute("load ducklake;")

con.sql(f"""
USE ducklake_bayesian;
""")

Attempting to automatically open the SSO authorization page in your default browser.
Please open this link to login into your account: https://auth.motherduck.com/activate?user_code=BKWM-QQHN


Token successfully retrieved ✅

You can display the token and store it as an environment variable to avoid having to log in again:
  PRAGMA PRINT_MD_TOKEN;


In [3]:
con.sql("DROP PROPERTY GRAPH IF EXISTS bayesian_kg")

con.sql("""
CREATE PROPERTY GRAPH bayesian_kg
  VERTEX TABLES (
    node_factor, node_disease, node_symptom
  )
EDGE TABLES (
  edge_causes 	SOURCE KEY ('from_id') REFERENCES node_factor (id)
                DESTINATION KEY ('to_id') REFERENCES node_disease (id)
  LABEL CAUSES,
        
  edge_cooccurs SOURCE KEY ('from_id') REFERENCES node_disease (id)
          DESTINATION KEY ('to_id') REFERENCES node_disease (id)
  LABEL COOCCURS,
  
  edge_disease_leads_to_factor SOURCE KEY ('from_id') REFERENCES node_disease (id)
          DESTINATION KEY ('to_id') REFERENCES node_factor (id)
  LABEL DISEASE_LEADS_TO_FACTOR,
        
  edge_has_symptom SOURCE KEY ('from_id') REFERENCES node_disease (id)
          DESTINATION KEY ('to_id') REFERENCES node_symptom (id)
  LABEL HAS_SYMPTOM,
  
  edge_leads_to_factor SOURCE KEY ('from_id') REFERENCES node_factor (id)
          DESTINATION KEY ('to_id') REFERENCES node_factor (id)
  LABEL LEADS_TO_FACTOR,

  edge_symptom_cooccurs SOURCE KEY ('from_id') REFERENCES node_symptom (id)
          DESTINATION KEY ('to_id') REFERENCES node_symptom (id)
  LABEL SYMPTOM_COOCCURS     
);
""")

┌─────────┐
│ Success │
│ boolean │
├─────────┤
│ 0 rows  │
└─────────┘

In [5]:
con.sql("""SELECT DISTINCT factor_name
  FROM GRAPH_TABLE (bayesian_kg
  MATCH
  (i:node_factor)-[m:CAUSES]->(c:node_disease WHERE c.name = 'COVID')
  COLUMNS (i.name AS factor_name)
  )
  LIMIT 30;
""")

┌─────────────┐
│ factor_name │
│   varchar   │
├─────────────┤
│ Travel      │
│ Smoking     │
└─────────────┘

In [6]:
con.close()